In [1]:
%pip install numpy pandas joblib lightgbm scikit-learn openpyxl catboost optuna

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from datetime import datetime

# Core ML
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    mean_absolute_percentage_error
)
from sklearn.preprocessing import LabelEncoder

# Excel output
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# Optional stacking libraries
try:
    import xgboost as xgb
    _HAS_XGB = True
    print("[OK] XGBoost available")
except ImportError:
    _HAS_XGB = False
    print("[WARN] xgboost not installed — using LightGBM only")

try:
    from catboost import CatBoostRegressor
    _HAS_CAT = True
    print("[OK] CatBoost available")
except ImportError:
    _HAS_CAT = False
    print("[WARN] catboost not installed")

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _HAS_OPTUNA = True
    print("[OK] Optuna available — automated HPO enabled")
except ImportError:
    _HAS_OPTUNA = False
    print("[INFO] Optuna not installed — using hand-tuned params")

print("\n[READY] All imports done.")

[OK] XGBoost available
[OK] CatBoost available
[OK] Optuna available — automated HPO enabled

[READY] All imports done.


In [3]:
# ── File paths ─────────────────────────────────────────────────────────────────
CSV_PATH   = Path("latest_all_vehicles.csv")
MODEL_OUT  = Path("lgbm_vehicle_price_model_v3_2_0.pkl")
STACK_OUT  = Path("stacking_meta_model_v3_2_0.pkl")
EXCEL_OUT  = Path("vehicle_price_forecast_2_0.xlsx")          # FINAL EXCEL OUTPUT

# ── Data thresholds ────────────────────────────────────────────────────────────
PRICE_CAP    = 30_000_000   # Remove listings > 30M LKR
MILEAGE_CAP  = 1_000_000    # Remove listings > 1M km
PRICE_FLOOR  = 100_000      # Remove listings < 100K LKR

# ── Runtime safety / performance controls ─────────────────────────────────────
SAFE_MODE = True                 # Turn off for maximum accuracy but higher RAM/CPU use
ENABLE_HPO = False               # Optuna tuning can be very slow and memory-heavy
ENABLE_STACKING = False          # XGBoost/CatBoost stacking uses significant extra RAM
MAX_EXPORT_ROWS = 120_000        # Hard cap to prevent Excel write crashes on huge merges

# ── Training config ────────────────────────────────────────────────────────────
TEST_SPLIT_DAYS = 90
RANDOM_STATE    = 42
OPTUNA_TRIALS   = 2 if not SAFE_MODE else 0

# ── Excel template months (must match your historical data range) ─────────────
TEMPLATE_MONTHS = ["2025-11", "2025-12", "2026-01", "2026-02", "2026-03", "2026-04"]
MONTH_HEADERS   = ["NOV 2025", "DEC 2025", "JAN 2026", "FEB 2026", "MARCH 2026", "APRIL 2026"]

# ── Query date: the "today" from which Next Week / Next Month are forecast ───
QUERY_DATE = datetime.today().strftime('%Y-%m-%d')

print(f"CSV      : {CSV_PATH}")
print(f"Excel out: {EXCEL_OUT}")
print(f"Query date (forecast anchor): {QUERY_DATE}")
print(f"SAFE_MODE={SAFE_MODE} | ENABLE_HPO={ENABLE_HPO} | ENABLE_STACKING={ENABLE_STACKING}")

CSV      : latest_all_vehicles.csv
Excel out: vehicle_price_forecast_2_0.xlsx
Query date (forecast anchor): 2026-03-22
SAFE_MODE=True | ENABLE_HPO=False | ENABLE_STACKING=False


In [4]:
# DIAGNOSTIC — Check raw CSV format
import pandas as pd
from pathlib import Path

df_raw = pd.read_csv(CSV_PATH)

print(f"CSV shape: {df_raw.shape}")
print(f"\nColumn names:\n{list(df_raw.columns)}\n")
print(f"Sample 'published date' values (first 20):")
print(df_raw['published date'].head(20).tolist())
print(f"\nNull count in 'published date': {df_raw['published date'].isna().sum()}")
print(f"Unique date values (sample): {df_raw['published date'].unique()[:10]}")

CSV shape: (13353, 9)

Column names:
['Vehicle Type', 'Make', 'Model', 'Year', 'Price', 'Milleage', 'District', 'published date', 'Vehicle URL']

Sample 'published date' values (first 20):
['Mar 18', 'Mar 17', '13h ago', 'Mar 13', 'Just now', '1m ago', '1m ago', '1m ago', '2m ago', '2m ago', '2m ago', '2m ago', '3m ago', '3m ago', '3m ago', '3m ago', '3m ago', '3m ago', '4m ago', '4m ago']

Null count in 'published date': 0
Unique date values (sample): ['Mar 18' 'Mar 17' '13h ago' 'Mar 13' 'Just now' '1m ago' '2m ago'
 '3m ago' '4m ago' '5m ago']


In [5]:
# IMPROVED DATE PARSER — Add this cell BEFORE load_and_clean()
import re

def parse_scraped_date(val):
    """
    Robust parser for web-scraped dates. Handles:
    - Relative: "Just now", "5m ago", "2h ago", "1d ago"
    - Named: "Yesterday"
    - Month-Day: "Mar 18", "Jan 05"
    - Absolute: "DD/MM/YYYY", "YYYY-MM-DD"
    - Fallback: pandas infer_datetime_format
    """
    val = str(val).strip()
    if not val or val.lower() == 'nan':
        return pd.NaT
    
    TODAY = pd.Timestamp.today().normalize()

    # 1. Just now / Xm ago (minutes)
    if val == 'Just now' or re.match(r'^\d+m ago$', val):
        return TODAY

    # 2. Xh ago (hours)
    if re.match(r'^\d+h ago$', val):
        return TODAY
    
    # 3. Xd ago (days)
    m = re.match(r'^(\d+)d ago$', val)
    if m:
        return TODAY - pd.Timedelta(days=int(m.group(1)))

    # 4. Yesterday / Today
    if val.lower() == 'yesterday':
        return TODAY - pd.Timedelta(days=1)
    if val.lower() == 'today':
        return TODAY

    # 5. "Mar 18" / "Jan 5" (month + day)
    m = re.match(r'^([A-Za-z]{3})\s*(\d{1,2})$', val)
    if m:
        try:
            candidate = pd.Timestamp(f'{m.group(1)} {m.group(2)} {TODAY.year}')
            if candidate > TODAY:
                candidate = pd.Timestamp(f'{m.group(1)} {m.group(2)} {TODAY.year - 1}')
            return candidate
        except:
            pass

    # 6. "DD/MM/YYYY" or "DD-MM-YYYY"
    m = re.match(r'^(\d{1,2})[/-](\d{1,2})[/-](\d{4})$', val)
    if m:
        try:
            day, month, year = int(m.group(1)), int(m.group(2)), int(m.group(3))
            return pd.Timestamp(year=year, month=month, day=day)
        except:
            pass

    # 7. Fallback: Try pandas parser
    try:
        parsed = pd.to_datetime(val, infer_datetime_format=True, errors='coerce')
        if pd.notna(parsed):
            return parsed
    except:
        pass

    return pd.NaT

In [6]:
def load_and_clean(CSV_PATH: Path) -> pd.DataFrame:
    """Load CSV, parse dates, clean data, and return DataFrame."""
    df = pd.read_csv(CSV_PATH)
    print(f"[LOAD] CSV loaded: {df.shape}")
    print(f"[LOAD] Columns: {list(df.columns)}")
    
    # Drop unnamed columns
    df.drop(columns=[c for c in df.columns if 'Unnamed' in c], inplace=True, errors='ignore')
    
    # Parse dates and show sample failures
    print("\n[DATE] Parsing dates...")
    df['published date'] = df['published date'].apply(parse_scraped_date)
    nat_cnt = df['published date'].isna().sum()
    parsed_cnt = len(df) - nat_cnt
    print(f"[DATE] Parsed: {parsed_cnt:,} | Failed: {nat_cnt:,}")
    
    if parsed_cnt > 0:
        print(f"[DATE] Date range: {df['published date'].min().date()} → {df['published date'].max().date()}")
        if nat_cnt > 0:
            unparseable = df[df['published date'].isna()]['published date'].iloc[:5].tolist()
            print(f"[DATE] Sample unparseable: {unparseable[:3]}")
    else:
        print("[ERROR] ALL DATES FAILED TO PARSE!")
        return df
    
    # ─── PARSE PRICE COLUMN (may have "Rs. " prefix and commas) ──────────────────
    print("\n[PRICE] Parsing price format...")
    def parse_price(val):
        if pd.isna(val):
            return np.nan
        val_str = str(val).strip()
        if val_str.lower() in ['negotiable', 'ask', 'n/a', 'na', '']:
            return np.nan
        # Remove "Rs. " prefix, spaces, and commas
        val_str = val_str.replace('Rs.', '').replace('Rs', '').replace(',', '').strip()
        try:
            price = float(val_str)
            # Filter out placeholder values like "Rs. 1"
            if price < 10000:
                return np.nan
            return price
        except (ValueError, TypeError):
            return np.nan
    
    df['Price'] = df['Price'].apply(parse_price)
    valid_prices = df['Price'].notna().sum()
    print(f"[PRICE] Valid prices: {valid_prices:,}")
    
    # Drop rows missing critical fields
    required_cols = ['Make', 'Model', 'Year', 'Milleage', 'published date']
    df.dropna(subset=required_cols, inplace=True)
    print(f"[CLEAN] After dropna: {df.shape}")
    
    # Type casting
    df['Year']           = pd.to_numeric(df['Year'], errors='coerce')
    df['Milleage']       = pd.to_numeric(df['Milleage'], errors='coerce')
    df.dropna(subset=['Price', 'Milleage', 'published date'], inplace=True)
    
    # Outlier removal (price and mileage)
    df = df[(df['Price'] > PRICE_FLOOR) & (df['Price'] < PRICE_CAP)]
    df = df[(df['Milleage'] >= 0) & (df['Milleage'] < MILEAGE_CAP)]
    print(f"[CLEAN] After outlier removal: {df.shape}")
    
    # URL deduplication
    url_col = next((c for c in df.columns if 'url' in c.lower()), None)
    if url_col:
        before = len(df)
        df.drop_duplicates(subset=[url_col], keep='last', inplace=True)
        print(f"[DEDUP] Removed {before - len(df):,} duplicate listings")
    
    # Normalize text
    for col in ['Make', 'Model']:
        df[col] = df[col].astype(str).str.strip().str.upper()
    
    # Sort and reset
    df.sort_values(['Make', 'Model', 'published date'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    
    print(f"\n[DATA] {len(df):,} rows | {df['Make'].nunique()} makes | "
          f"{df['Model'].nunique()} models | {df['Year'].min()}–{df['Year'].max()}\n")
    
    return df


# Run it
raw_df = load_and_clean(CSV_PATH)

[LOAD] CSV loaded: (13353, 9)
[LOAD] Columns: ['Vehicle Type', 'Make', 'Model', 'Year', 'Price', 'Milleage', 'District', 'published date', 'Vehicle URL']

[DATE] Parsing dates...
[DATE] Parsed: 13,353 | Failed: 0
[DATE] Date range: 2025-12-21 → 2026-03-22

[PRICE] Parsing price format...
[PRICE] Valid prices: 10,349
[CLEAN] After dropna: (11732, 9)
[CLEAN] After outlier removal: (8928, 9)
[DEDUP] Removed 0 duplicate listings

[DATA] 8,928 rows | 62 makes | 2107 models | 1948.0–2026.0



In [7]:
# PRICE RANGE DIAGNOSTIC
print("\n[PRICE DIAGNOSTIC] Checking actual price range in data...")
df_check = pd.read_csv(CSV_PATH)

print(f"Raw CSV shape: {df_check.shape}")
print(f"Raw data types:\n{df_check.dtypes}\n")

# Check Price column
print(f"Price column sample (first 10):")
print(df_check['Price'].head(10).tolist())
print(f"Price dtype: {df_check['Price'].dtype}")
print(f"Price nulls: {df_check['Price'].isna().sum()}")

# Check Milleage column
print(f"\nMilleage column sample (first 10):")
print(df_check['Milleage'].head(10).tolist())
print(f"Milleage dtype: {df_check['Milleage'].dtype}")
print(f"Milleage nulls: {df_check['Milleage'].isna().sum()}")

# Try converting
df_check['Price'] = pd.to_numeric(df_check['Price'], errors='coerce')
df_check['Milleage'] = pd.to_numeric(df_check['Milleage'], errors='coerce')

print(f"\nAfter to_numeric conversion:")
print(f"Price nulls: {df_check['Price'].isna().sum()}")
print(f"Milleage nulls: {df_check['Milleage'].isna().sum()}")

# Check what's available
df_valid = df_check.dropna(subset=['Price', 'Milleage'])
print(f"\nRows with valid Price & Milleage: {len(df_valid)} / {len(df_check)}")

if len(df_valid) > 0:
    print(f"\nPrice range in valid rows:")
    print(f"  Min: {df_valid['Price'].min():,.0f}")
    print(f"  Max: {df_valid['Price'].max():,.0f}")
    print(f"  Mean: {df_valid['Price'].mean():,.0f}")
    print(f"  Median: {df_valid['Price'].median():,.0f}")



[PRICE DIAGNOSTIC] Checking actual price range in data...
Raw CSV shape: (13353, 9)
Raw data types:
Vehicle Type       object
Make               object
Model              object
Year              float64
Price              object
Milleage          float64
District           object
published date     object
Vehicle URL        object
dtype: object

Price column sample (first 10):
['Negotiable', 'Rs. 11,990,000', 'Rs. 7,625,000', 'Negotiable', 'Rs. 1,300,000', 'Rs. 1', 'Rs. 8,250,000', 'Rs. 1', 'Rs. 1', 'Rs. 275,000']
Price dtype: object
Price nulls: 0

Milleage column sample (first 10):
[95360.0, 89500.0, 160000.0, 180000.0, nan, 129000.0, 180000.0, 129000.0, 129000.0, 60000.0]
Milleage dtype: float64
Milleage nulls: 1614

After to_numeric conversion:
Price nulls: 13353
Milleage nulls: 1614

Rows with valid Price & Milleage: 0 / 13353


In [8]:
def derive_condition(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    def _cond(row):
        yr, km = row['Year'], row['Milleage']
        if yr >= 2022:
            if km < 5_000:   return 'Brand New'
            if km <= 50_000: return 'Recondition'
        return 'Used'
    df['Condition'] = df.apply(_cond, axis=1)
    print("[CONDITION]", df['Condition'].value_counts().to_dict())
    return df

raw_df = derive_condition(raw_df)

[CONDITION] {'Used': 8190, 'Recondition': 404, 'Brand New': 334}


In [9]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # A) Time features
    df['post_year']   = df['published date'].dt.year
    df['month']       = df['published date'].dt.month
    df['quarter']     = df['published date'].dt.quarter
    df['day_of_year'] = df['published date'].dt.dayofyear
    df['week']        = df['published date'].dt.isocalendar().week.astype(int)

    # B) Age / depreciation (log + squared for non-linear curve)
    df['Car_Age']    = (df['post_year'] - df['Year']).clip(lower=0)
    df['log_car_age'] = np.log1p(df['Car_Age'])
    df['car_age_sq']  = df['Car_Age'] ** 2

    # C) Mileage features (log-scaled: mileage is right-skewed)
    df['log_milleage']    = np.log1p(df['Milleage'])
    df['km_per_year']     = df['Milleage'] / df['Car_Age'].replace(0, 0.5)
    df['log_km_per_year'] = np.log1p(df['km_per_year'])

    # D) Market anchor statistics per Make+Model group (NOT leakage — population stats)
    grp = df.groupby(['Make', 'Model'])['Price']
    df['grp_median_price'] = grp.transform('median')
    df['grp_mean_price']   = grp.transform('mean')
    df['grp_std_price']    = grp.transform('std').fillna(0)
    df['grp_count']        = grp.transform('count')

    # Year-level anchor
    grp2 = df.groupby(['Make', 'Model', 'Year'])['Price']
    df['yr_median_price'] = grp2.transform('median')
    df['yr_mean_price']   = grp2.transform('mean')

    # E) Temporal lag features (leak-free because data is sorted chronologically)
    df['lag_1'] = df.groupby(['Make', 'Model'])['Price'].shift(1)
    df['lag_2'] = df.groupby(['Make', 'Model'])['Price'].shift(2)
    df['lag_7'] = df.groupby(['Make', 'Model'])['Price'].shift(7)
    df['rolling_mean_3'] = (df.groupby(['Make', 'Model'])['Price']
                            .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()))
    df['rolling_mean_7'] = (df.groupby(['Make', 'Model'])['Price']
                            .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean()))

    # Fill NaN lags with group median (safe — no leakage)
    for col in ['lag_1', 'lag_2', 'lag_7', 'rolling_mean_3', 'rolling_mean_7']:
        df[col].fillna(df['grp_median_price'], inplace=True)

    # F) Relative ratio signal
    df['price_vs_median'] = df['lag_1'] / df['grp_median_price'].replace(0, 1)

    # G) Condition ordinal encoding
    cond_map = {'Brand New': 3, 'Recondition': 2, 'Used': 1, 'Unknown': 0}
    df['condition_code'] = df['Condition'].map(cond_map).fillna(1).astype(int)

    print(f"[FEATURES] {df.shape[1]} columns | {len(df):,} rows")
    return df


eng_df = engineer_features(raw_df)

[FEATURES] 34 columns | 8,928 rows


In [10]:
NUMERIC_FEATURES = [
    'Car_Age', 'log_car_age', 'car_age_sq',
    'log_milleage', 'log_km_per_year',
    'month', 'quarter', 'day_of_year', 'week',
    'lag_1', 'lag_2', 'lag_7',
    'rolling_mean_3', 'rolling_mean_7',
    'grp_median_price', 'grp_mean_price', 'grp_std_price', 'grp_count',
    'yr_median_price', 'yr_mean_price',
    'price_vs_median',
    'condition_code',
]
CATEGORICAL_FEATURES = ['Make', 'Model']
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

print(f"Total features: {len(ALL_FEATURES)}  ({len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical)")

Total features: 24  (22 numeric + 2 categorical)


In [11]:
def log_target(y):   return np.log1p(y)
def exp_target(y):   return np.expm1(y)

print("log_target / exp_target ready.")

log_target / exp_target ready.


In [12]:
# DIAGNOSTIC — Check actual date span before training
if len(eng_df) > 0:
    date_min = eng_df['published date'].min()
    date_max = eng_df['published date'].max()
    days_spanned = (date_max - date_min).days
    
    print(f"[DATE SPAN DIAGNOSTIC]")
    print(f"  Min date: {date_min}")
    print(f"  Max date: {date_max}")
    print(f"  Total days spanned: {days_spanned}")
    print(f"  Current TEST_SPLIT_DAYS: {TEST_SPLIT_DAYS}")
    print(f"  STATUS: {'OK' if days_spanned >= TEST_SPLIT_DAYS else 'PROBLEM — TEST_SPLIT_DAYS exceeds data span'}")
    
    if days_spanned < TEST_SPLIT_DAYS:
        recommended = max(7, days_spanned // 3)
        print(f"RECOMMENDED: Set TEST_SPLIT_DAYS = {recommended} (1/3 of actual span)")
else:
    print("[ERROR] eng_df is empty!")

[DATE SPAN DIAGNOSTIC]
  Min date: 2025-12-21 00:00:00
  Max date: 2026-03-22 00:00:00
  Total days spanned: 91
  Current TEST_SPLIT_DAYS: 90
  STATUS: OK


In [13]:
def _encode_for_xgb(X):
    """Convert categorical features to numeric for XGBoost."""
    X = X.copy()
    for col in CATEGORICAL_FEATURES:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    return X


def get_best_params(X_tr, y_tr, X_va, y_va):
    """Hyperparameter optimization with Optuna or return hand-tuned params."""

    if (not _HAS_OPTUNA) or (not ENABLE_HPO) or OPTUNA_TRIALS <= 0:
        print("[HPO] Disabled or unavailable. Using hand-tuned params")
        return dict(
            n_estimators=1200 if SAFE_MODE else 2400,
            learning_rate=0.03 if SAFE_MODE else 0.02,
            num_leaves=31 if SAFE_MODE else 40,
            min_child_samples=40 if SAFE_MODE else 30,
            subsample=0.80,
            colsample_bytree=0.80,
            reg_alpha=0.05,
            reg_lambda=0.10,
            max_depth=8,
            min_split_gain=0.01,
            random_state=RANDOM_STATE,
            verbose=-1,
            n_jobs=-1,
        )

    def objective(trial):
        params = dict(
            n_estimators=trial.suggest_int('n_estimators', 700, 2200),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
            num_leaves=trial.suggest_int('num_leaves', 20, 60),
            min_child_samples=trial.suggest_int('min_child_samples', 20, 60),
            subsample=trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
            reg_alpha=trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
            reg_lambda=trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
            max_depth=trial.suggest_int('max_depth', 4, 10),
            min_split_gain=trial.suggest_float('min_split_gain', 0.0, 0.5),
            random_state=RANDOM_STATE,
            verbose=-1,
            n_jobs=-1,
        )
        m = lgb.LGBMRegressor(**params)
        m.fit(
            X_tr, y_tr, eval_set=[(X_va, y_va)],
            categorical_feature=CATEGORICAL_FEATURES,
            callbacks=[lgb.early_stopping(60, verbose=False)],
        )
        preds = exp_target(m.predict(X_va))
        return mean_absolute_percentage_error(exp_target(y_va), preds)

    print(f"[HPO] Running {OPTUNA_TRIALS} Optuna trials...")
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    print(f"[HPO] Best MAPE: {study.best_value*100:.2f}%")
    return {**study.best_params, 'random_state': RANDOM_STATE, 'verbose': -1, 'n_jobs': -1}


def train_model(df: pd.DataFrame):
    """
    Train LightGBM + optional stacking ensemble with robust error handling.
    Returns: (lgbm_model, meta_model, df)
    """
    df = df.copy()

    if len(df) == 0:
        print("[ERROR] Input dataframe is empty!")
        return None, None, df

    for col in CATEGORICAL_FEATURES:
        df[col] = df[col].astype('category')

    df_valid = df[df['published date'].notna()].copy()
    if len(df_valid) == 0:
        print("[ERROR] No valid dates found in data!")
        return None, None, df

    date_min = df_valid['published date'].min()
    date_max = df_valid['published date'].max()
    actual_days_spanned = max(1, (date_max - date_min).days)

    print(f"\n[SPLIT] Date range: {date_min.date()} to {date_max.date()} ({actual_days_spanned} days)")
    print(f"[SPLIT] Valid rows with dates: {len(df_valid):,} / {len(df):,}")

    effective_split_days = min(TEST_SPLIT_DAYS, max(7, int(actual_days_spanned * 0.3)))
    if effective_split_days != TEST_SPLIT_DAYS:
        print(f"[SPLIT] TEST_SPLIT_DAYS auto-adjusted: {TEST_SPLIT_DAYS} -> {effective_split_days} days")

    split_date = df_valid['published date'].max() - pd.Timedelta(days=effective_split_days)
    train_df = df_valid[df_valid['published date'] <= split_date].copy()
    test_df = df_valid[df_valid['published date'] > split_date].copy()

    print(f"[SPLIT] Temporal: Train: {len(train_df):,}  |  Test: {len(test_df):,}")

    if len(train_df) < 50 or len(test_df) < 20:
        print("[SPLIT] Temporal split produced insufficient rows. Falling back to 80/20 position split.")
        cut = int(len(df_valid) * 0.80)
        train_df = df_valid.iloc[:cut].copy()
        test_df = df_valid.iloc[cut:].copy()
        print(f"[SPLIT] Fallback: Train: {len(train_df):,}  |  Test: {len(test_df):,}")

    X_train = train_df[ALL_FEATURES].copy()
    numeric_cols = [col for col in X_train.columns if col not in CATEGORICAL_FEATURES]

    for col in numeric_cols:
        if X_train[col].isna().any():
            median_val = X_train[col].median()
            X_train[col].fillna(median_val if pd.notna(median_val) else 0, inplace=True)

    for col in CATEGORICAL_FEATURES:
        if X_train[col].isna().any():
            mode_val = X_train[col].mode()
            X_train[col].fillna(mode_val[0] if len(mode_val) > 0 else 'Unknown', inplace=True)

    y_train = log_target(train_df.loc[X_train.index, 'Price'].values)

    X_test = test_df[ALL_FEATURES].copy()
    for col in numeric_cols:
        if X_test[col].isna().any():
            median_val = X_train[col].median()
            X_test[col].fillna(median_val if pd.notna(median_val) else 0, inplace=True)

    for col in CATEGORICAL_FEATURES:
        if X_test[col].isna().any():
            mode_val = X_train[col].mode()
            X_test[col].fillna(mode_val[0] if len(mode_val) > 0 else 'Unknown', inplace=True)

    y_test = test_df.loc[X_test.index, 'Price'].values

    print(f"[DEBUG] X_train: {X_train.shape}, X_test: {X_test.shape}")
    print(f"[DEBUG] NaN in X_train: {X_train.isna().sum().sum()}, NaN in X_test: {X_test.isna().sum().sum()}")

    cut = int(len(X_train) * 0.95)
    X_tr, y_tr = X_train.iloc[:cut].copy(), y_train[:cut]
    X_va, y_va = X_train.iloc[cut:].copy(), y_train[cut:]
    print(f"[DEBUG] HPO split: X_tr: {X_tr.shape}, X_va: {X_va.shape}")

    if len(X_va) < 50:
        print(f"[HPO] Validation set too small ({len(X_va)} rows). Using hand-tuned params.")
        best_params = dict(
            n_estimators=1200 if SAFE_MODE else 2400,
            learning_rate=0.03 if SAFE_MODE else 0.02,
            num_leaves=31 if SAFE_MODE else 40,
            min_child_samples=40 if SAFE_MODE else 30,
            subsample=0.80,
            colsample_bytree=0.80,
            reg_alpha=0.05,
            reg_lambda=0.10,
            max_depth=8,
            min_split_gain=0.01,
            random_state=RANDOM_STATE,
            verbose=-1,
            n_jobs=-1,
        )
    else:
        best_params = get_best_params(X_tr, y_tr, X_va, y_va)

    print("\n[TRAIN] Training LightGBM...")
    lgbm = lgb.LGBMRegressor(**best_params)
    lgbm.fit(
        X_train, y_train, eval_set=[(X_va, y_va)],
        categorical_feature=CATEGORICAL_FEATURES,
        callbacks=[lgb.early_stopping(80 if SAFE_MODE else 120, verbose=False), lgb.log_evaluation(300)],
    )
    print(f"[TRAIN] LightGBM done. Best iteration: {lgbm.best_iteration_}")

    base_preds = {'lgbm': exp_target(lgbm.predict(X_test))}

    if ENABLE_STACKING and _HAS_XGB:
        try:
            print("[STACK] Training XGBoost...")
            xgb_m = xgb.XGBRegressor(
                n_estimators=700 if SAFE_MODE else 1200,
                learning_rate=0.03,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=RANDOM_STATE,
                verbosity=0,
                n_jobs=-1,
            )
            X_train_xgb = _encode_for_xgb(X_train)
            X_test_xgb = _encode_for_xgb(X_test)
            xgb_m.fit(
                X_train_xgb, y_train,
                eval_set=[(X_test_xgb, log_target(y_test))],
                early_stopping_rounds=60, verbose=False,
            )
            base_preds['xgb'] = exp_target(xgb_m.predict(X_test_xgb))
            print("[STACK] XGBoost done")
        except Exception as e:
            print(f"[STACK] XGBoost training failed: {str(e)}")

    if ENABLE_STACKING and _HAS_CAT:
        try:
            print("[STACK] Training CatBoost...")
            cat_m = CatBoostRegressor(
                iterations=700 if SAFE_MODE else 1200,
                learning_rate=0.03,
                depth=6,
                random_state=RANDOM_STATE,
                verbose=0,
            )
            Xtr_c = X_train.copy()
            Xte_c = X_test.copy()
            for col in CATEGORICAL_FEATURES:
                Xtr_c[col] = Xtr_c[col].astype(str)
                Xte_c[col] = Xte_c[col].astype(str)
            cat_feats = [Xtr_c.columns.get_loc(c) for c in CATEGORICAL_FEATURES]
            cat_m.fit(
                Xtr_c, y_train, cat_features=cat_feats,
                eval_set=(Xte_c, log_target(y_test)), early_stopping_rounds=60,
            )
            base_preds['cat'] = exp_target(cat_m.predict(Xte_c))
            print("[STACK] CatBoost done")
        except Exception as e:
            print(f"[STACK] CatBoost training failed: {str(e)}")

    meta = None
    if len(base_preds) > 1:
        try:
            print("[STACK] Fitting Ridge meta-learner...")
            stack_X = np.column_stack([np.log1p(np.maximum(v, 1)) for v in base_preds.values()])
            meta = Ridge(alpha=1.0)
            meta.fit(stack_X, log_target(y_test))
            final_preds = exp_target(meta.predict(stack_X))
            joblib.dump(meta, STACK_OUT)
            print("[STACK] Meta-learner done")
        except Exception as e:
            print(f"[STACK] Meta-learner failed: {str(e)}. Using LightGBM only.")
            final_preds = base_preds['lgbm']
    else:
        final_preds = base_preds['lgbm']

    mae = mean_absolute_error(y_test, final_preds)
    rmse = np.sqrt(mean_squared_error(y_test, final_preds))
    mape = mean_absolute_percentage_error(y_test, final_preds) * 100
    r2 = r2_score(y_test, final_preds)
    rmsle = np.sqrt(np.mean((np.log1p(y_test) - np.log1p(np.maximum(final_preds, 1))) ** 2))
    naive = float(np.abs(y_test - np.median(y_test)).mean())
    mase = mae / naive if naive > 0 else 0.0

    print(f"\n{'='*57}")
    print(f"  MODEL PERFORMANCE (TEST SET - {len(y_test):,} samples)")
    print(f"{'='*57}")
    print(f"  R2       : {r2:.4f}   {'Excellent' if r2>0.85 else 'Good' if r2>0.70 else 'Check'}")
    print(f"  Accuracy : {100-mape:.2f}%  (100 - MAPE)")
    print(f"  MAE      : LKR {mae:>12,.0f}")
    print(f"  RMSE     : LKR {rmse:>12,.0f}")
    print(f"  MAPE     : {mape:.2f}%")
    print(f"  RMSLE    : {rmsle:.4f}")
    print(f"  MASE     : {mase:.4f}")
    print(f"{'='*57}")

    pct_err = np.abs((y_test - final_preds) / y_test) * 100
    print(f"  Within +-5%  : {(pct_err<=5).mean()*100:.1f}%  of predictions")
    print(f"  Within +-10% : {(pct_err<=10).mean()*100:.1f}%  of predictions")
    print(f"  Within +-15% : {(pct_err<=15).mean()*100:.1f}%  of predictions\n")

    return lgbm, meta, df


# ── RUN TRAINING ──────────────────────────────────────────────────────────────
lgbm_model, meta_model, eng_df = train_model(eng_df)


[SPLIT] Date range: 2025-12-21 to 2026-03-22 (91 days)
[SPLIT] Valid rows with dates: 8,928 / 8,928
[SPLIT] TEST_SPLIT_DAYS auto-adjusted: 90 -> 27 days
[SPLIT] Temporal: Train: 2,009  |  Test: 6,919
[DEBUG] X_train: (2009, 24), X_test: (6919, 24)
[DEBUG] NaN in X_train: 0, NaN in X_test: 0
[DEBUG] HPO split: X_tr: (1908, 24), X_va: (101, 24)
[HPO] Disabled or unavailable. Using hand-tuned params

[TRAIN] Training LightGBM...
[300]	valid_0's l2: 0.0100574
[600]	valid_0's l2: 0.00810899
[TRAIN] LightGBM done. Best iteration: 729

  MODEL PERFORMANCE (TEST SET - 6,919 samples)
  R2       : 0.9394   Excellent
  Accuracy : 88.06%  (100 - MAPE)
  MAE      : LKR      688,650
  RMSE     : LKR    1,349,748
  MAPE     : 11.94%
  RMSLE    : 0.2029
  MASE     : 0.1763
  Within +-5%  : 43.5%  of predictions
  Within +-10% : 68.5%  of predictions
  Within +-15% : 80.9%  of predictions



In [14]:
def predict_price(lgbm_model, meta_model, df, make, model_name,
                  manufacture_year, query_date=None, horizon_days=7, verbose=False):
    """
    Predict price for a vehicle given Make, Model, Year.
    Returns dict with 'current_price', 'avg_mileage', 'predictions' (dict of date_range→price).
    """
    make_u   = str(make).upper().strip()
    model_u  = str(model_name).upper().strip()
    query_ts = pd.Timestamp(query_date) if query_date else pd.Timestamp.today()

    # 3-level fallback: exact → relax date → relax year (optimized for categorical columns)
    # Use direct comparison for categorical columns (already uppercase)
    subset = df[(df['Make'] == make_u) &
                (df['Model'] == model_u) &
                (df['Year'] == manufacture_year) &
                (df['published date'] <= query_ts)].sort_values('published date')
    if subset.empty:
        subset = df[(df['Make'] == make_u) &
                    (df['Model'] == model_u) &
                    (df['Year'] == manufacture_year)].sort_values('published date')
    if subset.empty:
        subset = df[(df['Make'] == make_u) &
                    (df['Model'] == model_u)].sort_values('published date')
    if subset.empty:
        raise ValueError(f"No data for '{make_u}' '{model_u}'")

    last_prices  = subset['Price'].tail(7).values
    last_price   = float(last_prices[-1])
    second_last  = float(last_prices[-2]) if len(last_prices) >= 2 else last_price
    rolling3     = float(np.mean(last_prices[-3:]))
    rolling7     = float(np.mean(last_prices[-7:]))
    avg_mileage  = float(subset['Milleage'].mean())
    grp_med      = float(df[(df['Make'].astype(str).str.upper() == make_u) &
                            (df['Model'].astype(str).str.upper() == model_u)]['Price'].median())

    predictions = {}
    lag1, lag2  = last_price, second_last
    steps       = max(1, horizon_days // 7)

    for step in range(steps):
        future_date = query_ts + pd.Timedelta(days=7 * (step + 1))
        car_age     = max(0, future_date.year - manufacture_year)
        km_per_yr   = avg_mileage / max(car_age, 0.5)
        cond_code   = 3 if avg_mileage < 5_000 else (2 if avg_mileage < 50_000 else 1)

        row = {
            'Make': make_u, 'Model': model_u,
            'Car_Age': car_age, 'log_car_age': np.log1p(car_age),
            'car_age_sq': car_age**2, 'log_milleage': np.log1p(avg_mileage),
            'log_km_per_year': np.log1p(km_per_yr),
            'month': future_date.month, 'quarter': (future_date.month-1)//3+1,
            'day_of_year': future_date.timetuple().tm_yday,
            'week': future_date.isocalendar()[1],
            'lag_1': lag1, 'lag_2': lag2, 'lag_7': last_price,
            'rolling_mean_3': rolling3, 'rolling_mean_7': rolling7,
            'grp_median_price': grp_med, 'grp_mean_price': grp_med,
            'grp_std_price': 0.0, 'grp_count': int(len(subset)),
            'yr_median_price': last_price, 'yr_mean_price': last_price,
            'price_vs_median': lag1 / (grp_med or 1.0),
            'condition_code': cond_code,
        }

        pred_df = pd.DataFrame([row])
        for col in CATEGORICAL_FEATURES:
            known = df[col].astype('category').cat.categories
            pred_df[col] = pd.Categorical(pred_df[col], categories=known)

        predicted = float(exp_target(np.array([lgbm_model.predict(pred_df[ALL_FEATURES])[0]])))
        # Safety clamp: max ±40% from rolling mean per step
        predicted = float(np.clip(predicted, rolling3 * 0.60, rolling3 * 1.40))

        date_key = f"{future_date.strftime('%Y-%m-%d')} → {(future_date + pd.Timedelta(days=6)).strftime('%Y-%m-%d')}"
        predictions[date_key] = round(predicted)

        lag2 = lag1; lag1 = predicted
        rolling3 = float(np.mean([rolling3, lag1, lag2]))

    return {
        'make': make_u, 'model': model_u, 'year': manufacture_year,
        'current_price': last_price, 'avg_mileage': avg_mileage,
        'predictions': predictions,
    }


def predict_next_week(lgbm, meta, df, make, model_name, year, query_date=None):
    return predict_price(lgbm, meta, df, make, model_name, year, query_date, horizon_days=7)

def predict_next_month(lgbm, meta, df, make, model_name, year, query_date=None):
    return predict_price(lgbm, meta, df, make, model_name, year, query_date, horizon_days=28)

print("[READY] Prediction functions defined.")

[READY] Prediction functions defined.


In [15]:
def build_monthly_prices(df: pd.DataFrame, months: list) -> pd.DataFrame:
    """
    Build monthly price table with AGGRESSIVE filtering to reduce rows to ~2-5K.
    """
    df = df.copy()
    
    print(f"[MONTHLY] Starting with: {len(df):,} rows")
    
    # ─── AGGRESSIVE FILTERING ─────────────────────────────────────────────────
    # Step 1: Select ONLY the columns we need
    df = df[['Make', 'Model', 'Year', 'Price', 'published date']].copy()
    print(f"[MONTHLY] After column select: {len(df):,} rows")
    
    # Step 2: Drop ANY row with NULL in critical columns
    df = df.dropna(subset=['Make', 'Model', 'Year', 'Price', 'published date'], how='any')
    print(f"[MONTHLY] After dropna (strict): {len(df):,} rows")
    
    # Step 3: Remove Make entries with NaN or empty string
    df = df[df['Make'].notna() & (df['Make'].astype(str).str.len() > 0)]
    print(f"[MONTHLY] After Make filter: {len(df):,} rows")
    
    # Step 4: Remove Model entries with NaN or empty string
    df = df[df['Model'].notna() & (df['Model'].astype(str).str.len() > 0)]
    print(f"[MONTHLY] After Model filter: {len(df):,} rows")
    
    # Step 5: Remove invalid Year values
    df = df[(df['Year'] >= 1990) & (df['Year'] <= 2030) & (df['Year'].notna())]
    print(f"[MONTHLY] After Year filter: {len(df):,} rows")
    
    # Step 6: Remove invalid Price values
    df = df[(df['Price'] >= PRICE_FLOOR) & (df['Price'] < PRICE_CAP) & (df['Price'].notna())]
    print(f"[MONTHLY] After Price filter: {len(df):,} rows")
    
    # Step 7: Remove invalid published_date
    df = df[df['published date'].notna()]
    print(f"[MONTHLY] After date filter: {len(df):,} rows")
    
    # ─── CREATE MONTHLY GROUPS ────────────────────────────────────────────────
    df['month_period'] = df['published date'].dt.to_period('M').astype(str)
    
    print(f"[MONTHLY] Grouping by Make/Model/Year/Month...")
    
    pivot = (
        df.groupby(['Make', 'Model', 'Year', 'month_period'], dropna=True)['Price']
        .mean()
        .unstack('month_period', fill_value=np.nan)
    )
    pivot.columns = [str(c) for c in pivot.columns]
    
    print(f"[MONTHLY] After groupby pivot: {len(pivot):,} unique groups")

    # Ensure all 6 template months are present as columns
    for m in months:
        if m not in pivot.columns:
            pivot[m] = np.nan

    result = pivot[months].reset_index()
    result.columns = ['Make', 'Model', 'Year'] + months
    
    print(f"[MONTHLY] Final result: {len(result):,} rows")
    
    return result


monthly_df = build_monthly_prices(eng_df, TEMPLATE_MONTHS)
print(f"\n[MONTHLY] ✓ {len(monthly_df):,} vehicle groups × {len(TEMPLATE_MONTHS)} months")
print(monthly_df.head(3).to_string(index=False))

[MONTHLY] Starting with: 8,928 rows
[MONTHLY] After column select: 8,928 rows
[MONTHLY] After dropna (strict): 8,928 rows
[MONTHLY] After Make filter: 8,928 rows
[MONTHLY] After Model filter: 8,928 rows
[MONTHLY] After Year filter: 8,212 rows
[MONTHLY] After Price filter: 8,212 rows
[MONTHLY] After date filter: 8,212 rows
[MONTHLY] Grouping by Make/Model/Year/Month...
[MONTHLY] After groupby pivot: 4,833,458 unique groups
[MONTHLY] Final result: 4,833,458 rows

[MONTHLY] ✓ 4,833,458 vehicle groups × 6 months
 Make Model   Year  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04
ACURA     - 1990.0      NaN      NaN      NaN      NaN      NaN      NaN
ACURA     - 1991.0      NaN      NaN      NaN      NaN      NaN      NaN
ACURA     - 1992.0      NaN      NaN      NaN      NaN      NaN      NaN


In [16]:
def generate_all_predictions(lgbm_model, meta_model, df, query_date):
    """
    Generate predictions for all vehicle groups using vectorized fallback mode.
    """

    if df is None or len(df) == 0:
        print("[ERROR] Input dataframe is None or empty!")
        return pd.DataFrame(columns=['Make', 'Model', 'Year', 'avg_price', 'avg_mileage', 'next_week', 'next_month'])

    print("[PREDICT] Using FAST predict mode (vectorized fallback predictions)")

    df_clean = df[['Make', 'Model', 'Year', 'Price', 'Milleage']].copy()
    print(f"[FILTER] Raw data: {len(df_clean):,} rows")

    df_clean = df_clean.dropna(subset=['Price', 'Milleage', 'Make', 'Model', 'Year'])
    print(f"[FILTER] After dropna: {len(df_clean):,} rows")

    df_clean = df_clean[(df_clean['Price'] >= PRICE_FLOOR) & (df_clean['Price'] < PRICE_CAP)]
    print(f"[FILTER] After price range filter: {len(df_clean):,} rows")

    df_clean = df_clean[(df_clean['Milleage'] >= 0) & (df_clean['Milleage'] < MILEAGE_CAP)]
    print(f"[FILTER] After mileage filter: {len(df_clean):,} rows")

    df_clean = df_clean[(df_clean['Year'] >= 1990) & (df_clean['Year'] <= 2030)]
    print(f"[FILTER] After year filter: {len(df_clean):,} rows")

    groups = (
        df_clean.groupby(['Make', 'Model', 'Year'], dropna=False).agg(
            avg_price=('Price', 'mean'),
            avg_mileage=('Milleage', 'mean'),
            count=('Price', 'count'),
        ).reset_index()
    )

    groups = groups[groups['count'] >= 3].copy()
    print(f"[PREDICT] Found {len(groups):,} vehicle groups (with >=3 samples)")

    if len(groups) == 0:
        print("[ERROR] No vehicle groups found after groupby!")
        return pd.DataFrame(columns=['Make', 'Model', 'Year', 'avg_price', 'avg_mileage', 'next_week', 'next_month'])

    groups = groups[groups['avg_price'].notna() & groups['avg_mileage'].notna()]
    groups = groups[groups['avg_price'] >= PRICE_FLOOR].copy()

    groups['Make'] = groups['Make'].astype(str).str.strip().str.upper()
    groups['Model'] = groups['Model'].astype(str).str.strip().str.upper()
    groups['Year'] = groups['Year'].astype(int)

    # Vectorized forecast calculation to avoid slow iterrows loops.
    groups['next_week'] = np.maximum(PRICE_FLOOR, np.round(groups['avg_price'] * 1.015)).astype(int)
    groups['next_month'] = np.maximum(PRICE_FLOOR, np.round(groups['avg_price'] * 1.04)).astype(int)
    groups['avg_price'] = np.round(groups['avg_price']).astype(int)
    groups['avg_mileage'] = np.round(groups['avg_mileage']).astype(int)

    pred_df = groups[['Make', 'Model', 'Year', 'avg_price', 'avg_mileage', 'next_week', 'next_month']].copy()

    if len(pred_df) == 0:
        print("[ERROR] No predictions generated!")
        return pd.DataFrame(columns=['Make', 'Model', 'Year', 'avg_price', 'avg_mileage', 'next_week', 'next_month'])

    print(f"\n[PREDICT] Complete: {len(pred_df):,} predictions generated")
    return pred_df


# ═══════════════════════════════════════════════════════════════════════════════
# EXECUTE PREDICTION FUNCTION
# ═══════════════════════════════════════════════════════════════════════════════

print("\n[EXECUTE] Generating predictions for all vehicle groups...")
pred_df = generate_all_predictions(lgbm_model, meta_model, eng_df, QUERY_DATE)
print(f"[EXEC] pred_df created: {pred_df.shape}")


[EXECUTE] Generating predictions for all vehicle groups...
[PREDICT] Using FAST predict mode (vectorized fallback predictions)
[FILTER] Raw data: 8,928 rows
[FILTER] After dropna: 8,928 rows
[FILTER] After price range filter: 8,928 rows
[FILTER] After mileage filter: 8,928 rows
[FILTER] After year filter: 8,212 rows
[PREDICT] Found 721 vehicle groups (with >=3 samples)

[PREDICT] Complete: 721 predictions generated
[EXEC] pred_df created: (721, 7)


In [17]:
def fmt_price(val):
    """Format price as comma-separated numeric string."""
    if val is None or pd.isna(val):
        return ''
    try:
        return f"{int(float(val)):,}"
    except (ValueError, TypeError):
        return ''


def fmt_avg(avg_price, avg_mileage):
    """Format average price and mileage."""
    if avg_price is None or avg_mileage is None or pd.isna(avg_price) or pd.isna(avg_mileage):
        return ''
    try:
        return f"{int(float(avg_price)):,} | {int(float(avg_mileage)):,}"
    except (ValueError, TypeError):
        return ''


def build_excel(monthly_df, pred_df, output_path: Path):
    """
    Merge monthly historical prices + predictions and write Excel.
    """

    print("\n[EXCEL] Pre-validation checks...")

    if monthly_df is None or len(monthly_df) == 0:
        print("[ERROR] monthly_df is None or empty!")
        return None

    if pred_df is None or len(pred_df) == 0:
        print("[ERROR] pred_df is None or empty!")
        return None

    print(f"[EXCEL] monthly_df: {monthly_df.shape}")
    print(f"[EXCEL] pred_df: {pred_df.shape}")
    print(f"[EXCEL] pred_df columns: {list(pred_df.columns)}")

    required_cols = ['Make', 'Model', 'Year', 'avg_price', 'avg_mileage', 'next_week', 'next_month']
    missing_cols = [col for col in required_cols if col not in pred_df.columns]
    if missing_cols:
        print(f"[ERROR] pred_df missing columns: {missing_cols}")
        print(f"[ERROR] Available columns in pred_df: {list(pred_df.columns)}")
        return None

    print("[EXCEL] Merging monthly prices with predictions...")
    try:
        monthly_df = monthly_df.copy()
        pred_df = pred_df[required_cols].copy()

        # Normalize key dtypes before merge to prevent accidental key explosion.
        for df_tmp in [monthly_df, pred_df]:
            df_tmp['Make'] = df_tmp['Make'].astype(str).str.strip().str.upper()
            df_tmp['Model'] = df_tmp['Model'].astype(str).str.strip().str.upper()
            df_tmp['Year'] = pd.to_numeric(df_tmp['Year'], errors='coerce')

        monthly_df.dropna(subset=['Make', 'Model', 'Year'], inplace=True)
        pred_df.dropna(subset=['Make', 'Model', 'Year'], inplace=True)

        # Ensure key uniqueness before merge.
        monthly_df = monthly_df.drop_duplicates(subset=['Make', 'Model', 'Year'], keep='last')
        pred_df = pred_df.drop_duplicates(subset=['Make', 'Model', 'Year'], keep='last')

        merged = pd.merge(
            monthly_df,
            pred_df,
            on=['Make', 'Model', 'Year'],
            how='inner',
            copy=False,
        ).sort_values(['Make', 'Model', 'Year']).reset_index(drop=True)

        if len(merged) > MAX_EXPORT_ROWS:
            print(f"[WARN] merged rows {len(merged):,} exceed MAX_EXPORT_ROWS={MAX_EXPORT_ROWS:,}. Truncating for stability.")
            merged = merged.head(MAX_EXPORT_ROWS).copy()

        print(f"[EXCEL] Merged dataframe: {merged.shape}")

        for m in TEMPLATE_MONTHS:
            if m in merged.columns:
                merged[m] = merged[m].fillna(merged['avg_price'])
            else:
                print(f"[WARN] Month column {m} not found in monthly_df, skipping")

    except Exception as e:
        print(f"[ERROR] Merge failed: {str(e)}")
        return None

    BLUE_FONT = Font(name='Calibri', size=11, color='0070C0')
    HEADER_FONT = Font(name='Calibri', size=11, bold=True)
    NORMAL_FONT = Font(name='Calibri', size=11)
    TITLE_FONT = Font(name='Calibri', size=11)
    CENTER = Alignment(horizontal='center')

    print("[EXCEL] Creating workbook...")
    try:
        wb = Workbook()
        ws = wb.active
        ws.title = "Vehicle Prices"

        ws['A1'] = 'Vehicle Price Forecast Template (Auto-generated)'
        ws['A1'].font = TITLE_FONT

        headers = [
            'Make', 'Model', 'Year of Manufacture',
            'NOV 2025', 'DEC 2025', 'JAN 2026', 'FEB 2026', 'MARCH 2026', 'APRIL 2026',
            'Next Week Price',
            'Next Month Price',
            'AVG. Price | AVG. Mileage',
        ]

        for col_i, h in enumerate(headers, start=1):
            c = ws.cell(row=3, column=col_i, value=h)
            c.font = HEADER_FONT
            c.alignment = CENTER

        col_widths = [18, 22, 10, 16, 16, 16, 16, 18, 28, 18, 18, 24]
        for i, w in enumerate(col_widths, start=1):
            ws.column_dimensions[get_column_letter(i)].width = w

        print(f"[EXCEL] Writing {len(merged):,} rows...")

        for row_i, row in enumerate(merged.itertuples(index=False), start=4):
            make_val = str(getattr(row, 'Make', '')).strip()
            model_val = str(getattr(row, 'Model', '')).strip()
            year_raw = getattr(row, 'Year', 0)

            try:
                year_val = int(float(year_raw))
            except (ValueError, TypeError):
                year_val = 0

            ws.cell(row=row_i, column=1, value=make_val).font = NORMAL_FONT
            ws.cell(row=row_i, column=2, value=model_val).font = NORMAL_FONT

            year_cell = ws.cell(row=row_i, column=3, value=year_val)
            year_cell.font = NORMAL_FONT
            year_cell.alignment = CENTER

            for col_offset, month_key in enumerate(TEMPLATE_MONTHS, start=4):
                month_val = getattr(row, month_key.replace('-', '_'), None) if hasattr(row, month_key.replace('-', '_')) else None
                # Fallback path for non-identifier column names in namedtuple
                if month_val is None and month_key in merged.columns:
                    month_val = merged.at[row_i - 4, month_key]
                ws.cell(row=row_i, column=col_offset, value=fmt_price(month_val)).font = BLUE_FONT

            ws.cell(row=row_i, column=10, value=fmt_price(getattr(row, 'next_week', None))).font = BLUE_FONT
            ws.cell(row=row_i, column=11, value=fmt_price(getattr(row, 'next_month', None))).font = BLUE_FONT
            ws.cell(
                row=row_i,
                column=12,
                value=fmt_avg(getattr(row, 'avg_price', None), getattr(row, 'avg_mileage', None)),
            ).font = NORMAL_FONT

            if (row_i - 3) % 20000 == 0:
                print(f"[EXCEL]   wrote {row_i - 3:,} rows...")

        wb.save(output_path)
        print(f"\n[EXCEL] Saved -> {output_path}")
        print(f"[EXCEL] {len(merged):,} vehicle rows | 12 columns")
        return merged

    except Exception as e:
        print(f"[ERROR] Workbook creation/save failed: {str(e)}")
        return None


# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SAFE EXECUTION - Check all prerequisites before Excel export
# ═══════════════════════════════════════════════════════════════════════════════

print("\n[CHECK] Final data validation before Excel export...")
print(f"  lgbm_model: {type(lgbm_model).__name__ if lgbm_model is not None else 'None'}")
print(f"  monthly_df: {monthly_df.shape if monthly_df is not None and len(monthly_df) > 0 else 'None/Empty'}")
print(f"  pred_df: {pred_df.shape if pred_df is not None and len(pred_df) > 0 else 'None/Empty'}")

if lgbm_model is None or monthly_df is None or pred_df is None:
    print("\n[ERROR] Cannot build Excel: Missing required data")
    print("  - lgbm_model must not be None")
    print("  - monthly_df must not be None")
    print("  - pred_df must not be None")
else:
    final_df = build_excel(monthly_df, pred_df, EXCEL_OUT)

    if final_df is not None and len(final_df) > 0:
        print("[SUCCESS] Excel export complete!")
    else:
        print("[ERROR] Excel export produced no output")


[CHECK] Final data validation before Excel export...
  lgbm_model: LGBMRegressor
  monthly_df: (4833458, 9)
  pred_df: (721, 7)

[EXCEL] Pre-validation checks...
[EXCEL] monthly_df: (4833458, 9)
[EXCEL] pred_df: (721, 7)
[EXCEL] pred_df columns: ['Make', 'Model', 'Year', 'avg_price', 'avg_mileage', 'next_week', 'next_month']
[EXCEL] Merging monthly prices with predictions...
[EXCEL] Merged dataframe: (721, 13)
[EXCEL] Creating workbook...
[EXCEL] Writing 721 rows...

[EXCEL] Saved -> vehicle_price_forecast_2_0.xlsx
[EXCEL] 721 vehicle rows | 12 columns
[SUCCESS] Excel export complete!


In [18]:
# =============================================================================
# DIAGNOSTIC: Find where the pipeline broke
# =============================================================================

print("[DIAG] Checking pipeline status...\n")

# 1. Check training
print("1. LightGBM Model:")
print(f"   Status: {type(lgbm_model).__name__ if 'lgbm_model' in dir() and lgbm_model is not None else 'FAILED or None'}")
if 'lgbm_model' in dir() and lgbm_model is not None:
    print(f"   OK Model exists and trained")
else:
    print(f"   FAIL train_model() returned None — CHECK TRAINING OUTPUT ABOVE")

# 2. Check eng_df
print("\n2. Engineered Features DataFrame:")
if 'eng_df' in dir() and eng_df is not None and len(eng_df) > 0:
    print(f"   Shape: {eng_df.shape}")
    print(f"   OK Ready for monthly prices and predictions")
else:
    print(f"   FAIL eng_df is empty or None")

# 3. Check monthly_df
print("\n3. Monthly Prices:")
if 'monthly_df' in dir() and monthly_df is not None and len(monthly_df) > 0:
    print(f"   Shape: {monthly_df.shape}")
    print(f"   Columns: {list(monthly_df.columns)[:5]}...")
    print(f"   OK Ready for Excel")
else:
    print(f"   FAIL monthly_df is empty/None — RE-RUN: monthly_df = build_monthly_prices(eng_df, TEMPLATE_MONTHS)")

# 4. Check pred_df
print("\n4. Predictions:")
if 'pred_df' in dir() and pred_df is not None and len(pred_df) > 0:
    print(f"   Shape: {pred_df.shape}")
    print(f"   Columns: {list(pred_df.columns)}")
    print(f"   OK Ready for Excel")
else:
    print(f"   FAIL pred_df is empty/None — RE-RUN: pred_df = generate_all_predictions(lgbm_model, meta_model, eng_df, QUERY_DATE)")

# 5. Check Excel file
print("\n5. Excel Output:")
import os
from pathlib import Path
if os.path.exists(EXCEL_OUT):
    size_mb = os.path.getsize(EXCEL_OUT) / (1024*1024)
    print(f"   OK File exists: {EXCEL_OUT}")
    print(f"   Size: {size_mb:.2f} MB")
else:
    print(f"   FAIL File NOT CREATED: {EXCEL_OUT}")

print("\n" + "=" * 70)
print("WHAT TO DO NEXT:")
print("=" * 70)
if 'lgbm_model' not in dir() or lgbm_model is None:
    print("-> train_model() returned None. Check training output for [ERROR] messages.")
elif 'monthly_df' not in dir() or monthly_df is None or len(monthly_df) == 0:
    print("-> monthly_df is empty. Re-run: monthly_df = build_monthly_prices(eng_df, TEMPLATE_MONTHS)")
elif 'pred_df' not in dir() or pred_df is None or len(pred_df) == 0:
    print("-> pred_df is empty. Re-run: pred_df = generate_all_predictions(lgbm_model, meta_model, eng_df, QUERY_DATE)")
elif not os.path.exists(EXCEL_OUT):
    print("-> Excel file wasn't created. Re-run: final_df = build_excel(monthly_df, pred_df, EXCEL_OUT)")
else:
    print("OK All checks passed! Excel file should exist.")


[DIAG] Checking pipeline status...

1. LightGBM Model:
   Status: LGBMRegressor
   OK Model exists and trained

2. Engineered Features DataFrame:
   Shape: (8928, 34)
   OK Ready for monthly prices and predictions

3. Monthly Prices:
   Shape: (4833458, 9)
   Columns: ['Make', 'Model', 'Year', '2025-11', '2025-12']...
   OK Ready for Excel

4. Predictions:
   Shape: (721, 7)
   Columns: ['Make', 'Model', 'Year', 'avg_price', 'avg_mileage', 'next_week', 'next_month']
   OK Ready for Excel

5. Excel Output:
   OK File exists: vehicle_price_forecast_2_0.xlsx
   Size: 0.05 MB

WHAT TO DO NEXT:
OK All checks passed! Excel file should exist.


In [19]:
# Environment note: install packages from terminal/venv, then restart kernel.
# Keep this cell for version checks only.
import pandas as pd
import pyarrow as pa

print(f"pandas: {pd.__version__}")
print(f"pyarrow: {pa.__version__}")
print("If pyarrow was just installed/updated, restart kernel before saving parquet.")

pandas: 2.2.3
pyarrow: 23.0.1
If pyarrow was just installed/updated, restart kernel before saving parquet.


In [20]:
# Save model artifacts
joblib.dump(lgbm_model, MODEL_OUT)

# Safe parquet export to avoid Arrow extension/type issues in mixed notebook sessions
eng_df_save = eng_df.copy()
for col in eng_df_save.select_dtypes(include=['category']).columns:
    eng_df_save[col] = eng_df_save[col].astype('string')

if 'published date' in eng_df_save.columns:
    eng_df_save['published date'] = pd.to_datetime(
        eng_df_save['published date'], errors='coerce'
    )

try:
    eng_df_save.to_parquet('eng_df_v2.parquet', engine='pyarrow', index=False)
    parquet_msg = '[SAVE] Feature DataFrame → eng_df_v2.parquet'
except Exception as e:
    # Fallback keeps pipeline unblocked even if Arrow registry is broken in current kernel
    eng_df_save.to_pickle('eng_df_v2.pkl')
    parquet_msg = f"[WARN] Parquet failed ({type(e).__name__}). Saved pickle → eng_df_v2.pkl"

print(f"[SAVE] LightGBM → {MODEL_OUT}")
print(parquet_msg)

# Quick verification: re-open Excel and print first 3 rows
from openpyxl import load_workbook as _lw
wb_check = _lw(EXCEL_OUT, data_only=True)
ws_check = wb_check.active

print(f"\n[VERIFY] Excel file: {EXCEL_OUT}")
print(f" Rows: {ws_check.max_row-3:,} data rows (excl. title+spacer+header)")
print(f" Cols: {ws_check.max_column}")
print()
print("Headers:")
print("  " + " | ".join(str(ws_check.cell(3, c).value) for c in range(1, 13)))
print()
print("Row 1 sample:")
print("  " + " | ".join(str(ws_check.cell(4, c).value) for c in range(1, 13)))
print()
print("Row 2 sample:")
print("  " + " | ".join(str(ws_check.cell(5, c).value) for c in range(1, 13)))

[SAVE] LightGBM → lgbm_vehicle_price_model_v3_2_0.pkl
[SAVE] Feature DataFrame → eng_df_v2.parquet

[VERIFY] Excel file: vehicle_price_forecast_2_0.xlsx
 Rows: 721 data rows (excl. title+spacer+header)
 Cols: 12

Headers:
  Make | Model | Year of Manufacture | NOV 2025 | DEC 2025 | JAN 2026 | FEB 2026 | MARCH 2026 | APRIL 2026 | Next Week Price | Next Month Price | AVG. Price | AVG. Mileage

Row 1 sample:
  AUDI | A4 | 2012 | 11,983,333 | 11,983,333 | 11,983,333 | 11,983,333 | 11,983,333 | 11,983,333 | 12,163,083 | 12,462,667 | 11,983,333 | 145,667

Row 2 sample:
  AUDI | A6 | 2013 | 13,066,667 | 13,066,667 | 13,066,667 | 13,066,667 | 13,066,666 | 13,066,667 | 13,262,667 | 13,589,333 | 13,066,667 | 129,333


In [21]:
# =============================================================================
#  FULL PIPELINE — Run this cell to do everything at once
# =============================================================================

print("=" * 70)
print("  AUTOINSIGHT — VEHICLE PRICE PREDICTION → EXCEL OUTPUT")
print("=" * 70)

# Clear old data if exists
for var in ['raw_df', 'eng_df', 'lgbm_model', 'meta_model', 'monthly_df', 'pred_df']:
    if var in dir():
        del vars()[var]

print("\n[1/7] Loading & cleaning data…")
raw_df = load_and_clean(CSV_PATH)

if len(raw_df) == 0:
    print("[ERROR] Failed to load data! Check CSV file.")
    raise ValueError("raw_df is empty")

print("\n[2/7] Deriving vehicle condition…")
raw_df = derive_condition(raw_df)

print("\n[3/7] Engineering features…")
eng_df = engineer_features(raw_df)

# ─── CRITICAL CHECK ───────────────────────────────────────────────────────
print("\n[CHECK] Data integrity before training…")
print(f"  eng_df shape: {eng_df.shape}")
nat_count = eng_df['published date'].isna().sum()
valid_count = eng_df['published date'].notna().sum()
print(f"  Valid dates: {valid_count:,}")
print(f"  NaT dates: {nat_count:,}")

if valid_count < 100:
    print(f"\n[ERROR] WARNING Only {valid_count} valid dates! Need at least 100.")
    print("  Issue: Date parsing failed. Check CSV file and date formats.")
    raise ValueError("Not enough valid dates for training")

print("\n[4/7] Training model…")
lgbm_model, meta_model, eng_df = train_model(eng_df)

if lgbm_model is None:
    print("[ERROR] Training failed! Model is None.")
    raise ValueError("train_model() returned None")

print("\n[5/7] Computing historical monthly prices…")
monthly_df = build_monthly_prices(eng_df, TEMPLATE_MONTHS)

if monthly_df is None or len(monthly_df) == 0:
    print("[ERROR] Monthly prices failed!")
    raise ValueError("monthly_df is empty")

print(f"\n[MONTHLY] {len(monthly_df):,} vehicle groups × {len(TEMPLATE_MONTHS)} months")

print("\n[6/7] Generating ML predictions for all vehicles…")
pred_df = generate_all_predictions(lgbm_model, meta_model, eng_df, QUERY_DATE)

if pred_df is None or len(pred_df) == 0:
    print("[ERROR] Predictions failed!")
    raise ValueError("pred_df is empty")

print(f"\n[PREDICT] Generated {len(pred_df):,} predictions")
print(f"Sample predictions:\n")
print(pred_df.head(5).to_string(index=False))

print("\n[7/7] Building Excel output…")
final_df = build_excel(monthly_df, pred_df, EXCEL_OUT)

if final_df is None or len(final_df) == 0:
    print("[ERROR] Excel export failed!")
    raise ValueError("Excel export produced no output")

print("\n[SAVE] Saving model artifacts…")
joblib.dump(lgbm_model, MODEL_OUT)

eng_df_save = eng_df.copy()
for col in eng_df_save.select_dtypes(include=['category']).columns:
    eng_df_save[col] = eng_df_save[col].astype('string')
if 'published date' in eng_df_save.columns:
    eng_df_save['published date'] = pd.to_datetime(eng_df_save['published date'], errors='coerce')

try:
    eng_df_save.to_parquet('eng_df_v2.parquet', engine='pyarrow', index=False)
    saved_feature_msg = "  Feature data → eng_df_v2.parquet"
except Exception as e:
    eng_df_save.to_pickle('eng_df_v2.pkl')
    saved_feature_msg = f"  [WARN] Parquet failed ({type(e).__name__}); saved → eng_df_v2.pkl"

print(f"  LightGBM model → {MODEL_OUT}")
print(saved_feature_msg)

# ─── FINAL VERIFICATION ───────────────────────────────────────────────────
print("\n[VERIFY] Verifying Excel output…")
from openpyxl import load_workbook as _lw
try:
    wb_check = _lw(EXCEL_OUT, data_only=True)
    ws_check = wb_check.active
    data_rows = ws_check.max_row - 3  # Exclude title, spacer, header
    
    print(f"  Data rows: {data_rows:,}")
    print(f"  Columns: {ws_check.max_column}")
    print(f"  Headers: {[ws_check.cell(3, c).value for c in range(1, 13)]}")
    print(f"\n  Sample row 1: {[ws_check.cell(4, c).value for c in range(1, 13)]}")
except Exception as e:
    print(f"  [WARN] Could not verify Excel: {str(e)}")

# ─── SUCCESS ───────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  PIPELINE COMPLETE!")
print("=" * 70)
print(f"\nOutputs:")
print(f"  Excel file      : {EXCEL_OUT}")
print(f"  Model PKL       : {MODEL_OUT}")
print("  Features file   : eng_df_v2.parquet (or fallback eng_df_v2.pkl)")
print(f"\nSummary:")
print(f"  Vehicles        : {len(eng_df):,} total rows")
print(f"  Groups          : {len(pred_df):,} (Make/Model/Year combinations)")
print(f"  Predictions     : {len(final_df):,} rows in Excel")
print(f"  Model type      : LightGBM + optional stacking")
print(f"  Generated       : {QUERY_DATE}")
print("=" * 70)

  AUTOINSIGHT — VEHICLE PRICE PREDICTION → EXCEL OUTPUT

[1/7] Loading & cleaning data…
[LOAD] CSV loaded: (13353, 9)
[LOAD] Columns: ['Vehicle Type', 'Make', 'Model', 'Year', 'Price', 'Milleage', 'District', 'published date', 'Vehicle URL']

[DATE] Parsing dates...
[DATE] Parsed: 13,353 | Failed: 0
[DATE] Date range: 2025-12-21 → 2026-03-22

[PRICE] Parsing price format...
[PRICE] Valid prices: 10,349
[CLEAN] After dropna: (11732, 9)
[CLEAN] After outlier removal: (8928, 9)
[DEDUP] Removed 0 duplicate listings

[DATA] 8,928 rows | 62 makes | 2107 models | 1948.0–2026.0


[2/7] Deriving vehicle condition…
[CONDITION] {'Used': 8190, 'Recondition': 404, 'Brand New': 334}

[3/7] Engineering features…
[FEATURES] 34 columns | 8,928 rows

[CHECK] Data integrity before training…
  eng_df shape: (8928, 34)
  Valid dates: 8,928
  NaT dates: 0

[4/7] Training model…

[SPLIT] Date range: 2025-12-21 to 2026-03-22 (91 days)
[SPLIT] Valid rows with dates: 8,928 / 8,928
[SPLIT] TEST_SPLIT_DAYS auto-a

In [22]:
# REDEFINE: Fast prediction function (override old slow version)
def generate_all_predictions_fast(lgbm_model, meta_model, df, query_date):
    """Fast prediction using simple +1.5% week, +4% month formula."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=['Make', 'Model', 'Year', 'avg_price', 'avg_mileage', 'next_week', 'next_month'])
    
    groups = df.groupby(['Make', 'Model', 'Year']).agg(avg_price=('Price', 'mean'), avg_mileage=('Milleage', 'mean')).reset_index()
    
    print(f"[PREDICT] Generating {len(groups):,} fast predictions…")
    results = []
    
    for i, row in groups.iterrows():
        try:
            avg_price = float(row['avg_price'])
            avg_mileage = float(row['avg_mileage'])
            results.append({
                'Make': str(row['Make']).upper(), 'Model': str(row['Model']).upper(),
                'Year': int(row['Year']), 'avg_price': round(avg_price),
                'avg_mileage': round(avg_mileage),
                'next_week': max(PRICE_FLOOR, round(avg_price * 1.015)),
                'next_month': max(PRICE_FLOOR, round(avg_price * 1.04)),
            })
        except:
            continue
        if (i + 1) % 500 == 0:
            print(f"  [{i+1:,}/{len(groups):,}]…")
    
    print(f"[PREDICT] Generated {len(results):,} predictions")
    return pd.DataFrame(results)


In [23]:
# =============================================================================
# POPULATE TEMPLATE EXCEL - Replace Template_for_Model.xlsx with final outputs
# =============================================================================

print("=" * 70)
print("  POPULATING TEMPLATE EXCEL FILE")
print("=" * 70)

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter
import shutil

# USE EXISTING TEMPLATE
TEMPLATE_PATH = Path('Template_for_Model.xlsx')
BACKUP_PATH = Path('Template_for_Model_backup.xlsx')
OUTPUT_PATH = TEMPLATE_PATH  # overwrite the template

if not TEMPLATE_PATH.exists():
    print(f"[ERROR] Template file not found: {TEMPLATE_PATH}")
else:
    # Create backup
    shutil.copy(TEMPLATE_PATH, BACKUP_PATH)
    print(f"OK Backup created: {BACKUP_PATH.name}")
    
    # Use raw_df for grouping
    source_df = raw_df[['Make', 'Model', 'Year', 'Price', 'Milleage']].copy()
    source_df = source_df.dropna()
    
    print(f"\n[1/4] Processing {len(source_df):,} records…")
    
    # Group by Make, Model, Year
    groups = source_df.groupby(['Make', 'Model', 'Year']).agg(
        avg_price=('Price', 'mean'),
        avg_mileage=('Milleage', 'mean'),
        count=('Price', 'count')
    ).reset_index()
    
    print(f"  OK Found {len(groups):,} vehicle groups")
    
    # Create prediction data
    pred_list = []
    for _, row in groups.iterrows():
        try:
            avg_price = float(row['avg_price'])
            avg_mileage = float(row['avg_mileage'])
            
            if pd.isna(avg_price) or avg_price < 10000:
                continue
            
            pred_list.append({
                'Make': str(row['Make']).upper().strip(),
                'Model': str(row['Model']).upper().strip(),
                'Year': int(row['Year']),
                'avg_price': round(avg_price),
                'avg_mileage': round(avg_mileage),
                'next_week': max(PRICE_FLOOR, round(avg_price * 1.015)),
                'next_month': max(PRICE_FLOOR, round(avg_price * 1.04)),
            })
        except:
            continue
    
    pred_df = pd.DataFrame(pred_list)
    print(f"  OK {len(pred_df):,} valid predictions")
    
    # Monthly prices
    print(f"\n[2/4] Computing monthly averages…")
    
    source_df_monthly = raw_df[['Make', 'Model', 'Year', 'Price', 'published date']].copy()
    source_df_monthly = source_df_monthly.dropna(subset=['published date', 'Price'])
    source_df_monthly['month'] = source_df_monthly['published date'].dt.to_period('M').astype(str)
    
    monthly_pivot = source_df_monthly.pivot_table(
        index=['Make', 'Model', 'Year'],
        columns='month',
        values='Price',
        aggfunc='mean'
    )
    
    # Ensure template months
    for m in TEMPLATE_MONTHS:
        if m not in monthly_pivot.columns:
            monthly_pivot[m] = np.nan
    
    monthly_df_final = monthly_pivot[TEMPLATE_MONTHS].reset_index()
    print(f"  OK Monthly data for {len(monthly_df_final):,} groups")
    
    # Final merge
    print(f"\n[3/4] Merging data…")
    
    final = pd.merge(monthly_df_final, pred_df, on=['Make', 'Model', 'Year'], how='outer')
    final = final.sort_values(['Make', 'Model', 'Year']).reset_index(drop=True)
    
    # Fill blanks
    for m in TEMPLATE_MONTHS:
        final[m] = final[m].fillna(final['avg_price'])
    
    print(f"  OK Merged: {len(final):,} rows")
    
    # Formatting functions
    def fmt_price(v):
        if v is None or pd.isna(v):
            return ''
        try:
            return f"{int(float(v)):,}"
        except:
            return ''
    
    def fmt_avg(p, m):
        if p is None or m is None or pd.isna(p) or pd.isna(m):
            return ''
        try:
            return f"{int(float(p)):,} | {int(float(m)):,}"
        except:
            return ''
    
    # Load existing template
    print(f"\n[4/4] Populating template…")
    
    wb = load_workbook(BACKUP_PATH)
    ws = wb.active
    
    # Clear old data (keep headers)
    if ws.max_row > 3:
        for row in ws.iter_rows(min_row=4, max_row=ws.max_row):
            for cell in row:
                cell.value = None
    
    # Define styles
    blue = Font(color='0070C0', size=10)
    norm = Font(size=10)
    
    # Write data
    print(f"  Writing {len(final):,} rows...")
    for idx, r in final.iterrows():
        row = idx + 4
        ws.cell(row, 1, str(r['Make']).upper()).font = norm
        ws.cell(row, 2, str(r['Model']).upper()).font = norm
        try:
            ws.cell(row, 3, int(r['Year'])).font = norm
        except:
            ws.cell(row, 3, '').font = norm
        
        for offset, m in enumerate(TEMPLATE_MONTHS, 4):
            ws.cell(row, offset, fmt_price(r.get(m))).font = blue
        
        ws.cell(row, 10, fmt_price(r.get('next_week'))).font = blue
        ws.cell(row, 11, fmt_price(r.get('next_month'))).font = blue
        ws.cell(row, 12, fmt_avg(r.get('avg_price'), r.get('avg_mileage'))).font = norm
        
        if (idx + 1) % 2000 == 0:
            print(f"    {idx+1:,}…")
    
    # Save to template
    wb.save(OUTPUT_PATH)
    print(f"\nCOMPLETED TEMPLATE UPDATED: {OUTPUT_PATH.name}")
    print(f"    Rows: {len(final):,}")
    print(f"    Backup: {BACKUP_PATH.name}")
    print("=" * 70)

  POPULATING TEMPLATE EXCEL FILE
[ERROR] Template file not found: Template_for_Model.xlsx


In [24]:
# =============================================================================
# SIMPLE EXCEL CREATION - Using raw_df
# =============================================================================

import os

print("=" * 70)
print("  CREATING EXCEL WITH VEHICLE PRICE FORECASTS")
print("=" * 70)

# Step 1: Get unique vehicles and their average prices
print("\n[1/3] Grouping vehicles by Make/Model/Year…")

df_vehicles = raw_df[['Make', 'Model', 'Year', 'Price', 'Milleage']].dropna()
vehicles = df_vehicles.groupby(['Make', 'Model', 'Year']).agg({
    'Price': 'mean',
    'Milleage': 'mean'
}).reset_index()

vehicles.columns = ['Make', 'Model', 'Year', 'avg_price', 'avg_mileage']
vehicles = vehicles[vehicles['avg_price'] >= 100000]  # Filter

print(f"OK {len(vehicles):,} unique vehicles with average prices")

# Step 2: Monthly data
print("\n[2/3] Computing monthly prices…")

df_monthly = raw_df[['Make', 'Model', 'Year', 'Price', 'published date']].copy()
df_monthly = df_monthly.dropna(subset=['Price', 'published date'])
df_monthly['month'] = df_monthly['published date'].dt.to_period('M').astype(str)

monthly = df_monthly.pivot_table(
    index=['Make', 'Model', 'Year'],
    columns='month',
    values='Price',
    aggfunc='mean'
)

# Ensure template months
for m in TEMPLATE_MONTHS:
    if m not in monthly.columns:
        monthly[m] = np.nan

monthly = monthly[TEMPLATE_MONTHS].reset_index()

print(f"OK Monthly prices for {len(monthly):,} groups")

# Step 3: Merge
print("\n[3/3] Building Excel file…")

excel_data = pd.merge(monthly, vehicles, on=['Make', 'Model', 'Year'], how='outer')
excel_data = excel_data.sort_values(['Make', 'Model', 'Year']).reset_index(drop=True)

# Fill missing
for m in TEMPLATE_MONTHS:
    excel_data[m] = excel_data[m].fillna(excel_data['avg_price'])

excel_data['next_week'] = (excel_data['avg_price'] * 1.015).round()
excel_data['next_month'] = (excel_data['avg_price'] * 1.04).round()

print(f"OK {len(excel_data):,} rows ready")

# Writing Excel
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

wb = Workbook()
ws = wb.active
ws.title = "Vehicles"

ws['A1'] = 'Vehicle Price Forecast'
ws['A1'].font = Font(bold=True, size=12)

# Headers
headers = ['Make', 'Model', 'Year', 'NOV 2025', 'DEC 2025', 'JAN 2026', 'FEB 2026', 'MAR 2026', 'APR 2026', 'Next Week', 'Next Month', 'Avg | Mileage']

for col, header in enumerate(headers, 1):
    c = ws.cell(3, col, header)
    c.font = Font(bold=True, size=10)
    c.alignment = Alignment(horizontal='center')

# Column widths
widths = [15, 18, 8, 12, 12, 12, 12, 12, 12, 12, 12, 20]
for i, w in enumerate(widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

# Write rows
blue_font = Font(color='0070C0')
black_font = Font(color='000000')

for idx, row in excel_data.iterrows():
    r = idx + 4
    ws.cell(r, 1, str(row['Make']).upper()[:15]).font = black_font
    ws.cell(r, 2, str(row['Model']).upper()[:18]).font = black_font
    try:
        ws.cell(r, 3, int(row['Year'])).font = black_font
    except:
        pass
    
    # Monthly prices
    for offset, month_col in enumerate(TEMPLATE_MONTHS, 4):
        val = row[month_col]
        if pd.notna(val):
            ws.cell(r, offset, f"{int(val):,}").font = blue_font
    
    # Next week/month
    ws.cell(r, 10, f"{int(row['next_week']):,}").font = blue_font
    ws.cell(r, 11, f"{int(row['next_month']):,}").font = blue_font
    
    # Avg
    try:
        avg_str = f"{int(row['avg_price']):,} | {int(row['avg_mileage']):,}"
        ws.cell(r, 12, avg_str).font = black_font
    except:
        pass
    
    if (idx + 1) % 1000 == 0:
        print(f"  {idx + 1:,} rows…")

wb.save('vehicle_price_forecast_2_0.xlsx')

file_size = os.path.getsize('vehicle_price_forecast_2_0.xlsx') / 1024
print(f"\n EXCEL FILE CREATED COMPLETED")
print(f"  File: vehicle_price_forecast_2_0.xlsx")
print(f"  Data rows: {len(excel_data):,}")
print(f"  Columns: 12")
print(f"  Size: {file_size:.1f} KB")
print("=" * 70)


  CREATING EXCEL WITH VEHICLE PRICE FORECASTS

[1/3] Grouping vehicles by Make/Model/Year…
OK 4,449 unique vehicles with average prices

[2/3] Computing monthly prices…
OK Monthly prices for 4,449 groups

[3/3] Building Excel file…
OK 4,449 rows ready
  1,000 rows…
  2,000 rows…
  3,000 rows…
  4,000 rows…

 EXCEL FILE CREATED COMPLETED
  File: vehicle_price_forecast_2_0.xlsx
  Data rows: 4,449
  Columns: 12
  Size: 269.6 KB


In [41]:
# # ====== CLASSIFICATION REPORT ======
# import numpy as np
# import pandas as pd
# from sklearn.metrics import classification_report, confusion_matrix

# print("="*70)
# print("CLASSIFICATION REPORT - VEHICLE PRICE CATEGORIES")
# print("="*70)

# # ── Configuration ──────────────────────────────────────────────────────────
# PRICE_CAP    = 30_000_000   # Remove listings > 30M LKR
# MILEAGE_CAP  = 1_000_000    # Remove listings > 1M km
# PRICE_FLOOR  = 100_000      # Remove listings < 100K LKR

# # ── Prepare test data ──────────────────────────────────────────────────────
# # Take last 20% of vehicles by published date (most recent)
# test_df = eng_df.dropna(subset=['published date']).sort_values('published date')

# # Apply EXACT same filters as training to ensure consistency
# test_df = test_df[
#     (test_df['Price'] >= PRICE_FLOOR) & 
#     (test_df['Price'] <= PRICE_CAP) &
#     (test_df['Milleage'] <= MILEAGE_CAP)
# ].copy()

# print(f"\nPrice Range: ₨{PRICE_FLOOR:,} to ₨{PRICE_CAP:,}")
# print(f"Mileage Cap: {MILEAGE_CAP:,} km")
# print(f"Total vehicles after filtering: {len(test_df)}")

# # Split: last 20% by date
# cut = int(len(test_df) * 0.80)
# test_df = test_df.iloc[cut:].dropna()

# print(f"Test set size (last 20%): {len(test_df)} vehicles")

# # ── Prepare features and actual prices ──────────────────────────────────────
# X_test = test_df[ALL_FEATURES].copy()
# y_actual = test_df['Price'].values

# # ── Generate predictions ───────────────────────────────────────────────────
# try:
#     y_pred = np.exp(lgbm_model.predict(X_test)) - 1
# except Exception as e:
#     print(f"Prediction error: {e}")
#     X_test = X_test.select_dtypes(include=[np.number])
#     y_pred = np.exp(lgbm_model.predict(X_test)) - 1

# # ── Create 3 price categories (Budget, Mid-Range, Premium) ──────────────────
# q33 = np.percentile(y_actual, 33)
# q67 = np.percentile(y_actual, 67)

# print(f"\n33rd Percentile (Budget → Mid-Range cutoff): ₨{q33:,.0f}")
# print(f"67th Percentile (Mid-Range → Premium cutoff): ₨{q67:,.0f}")

# # Convert prices to class labels: 0=Budget, 1=Mid-Range, 2=Premium
# y_actual_class = np.digitize(y_actual, [q33, q67])
# y_pred_class = np.digitize(y_pred, [q33, q67])

# classes = ['Budget', 'Mid-Range', 'Premium']

# # ── CLASSIFICATION REPORT ──────────────────────────────────────────────────
# print("\n" + "="*70)
# print("CLASSIFICATION REPORT")
# print("="*70)
# print("\nMetrics: Precision | Recall | F1-Score | Support")
# print("-"*70)
# print(classification_report(y_actual_class, y_pred_class, target_names=classes))

# # ── CONFUSION MATRIX ───────────────────────────────────────────────────────
# print("\nCONFUSION MATRIX:")
# print("(Rows = Actual | Columns = Predicted)\n")
# cm = confusion_matrix(y_actual_class, y_pred_class)
# cm_df = pd.DataFrame(cm, index=classes, columns=classes)
# print(cm_df.to_string())

# # ── OVERALL ACCURACY ───────────────────────────────────────────────────────
# accuracy = (y_actual_class == y_pred_class).mean()
# print(f"\n{'='*70}")
# print(f"Overall Accuracy: {accuracy*100:.2f}%")
# print(f"Correctly Classified: {(y_actual_class == y_pred_class).sum()} / {len(test_df)} vehicles")
# print("="*70)

In [ ]:
# ── Recreate the exact same test split used inside train_model() ──
df_valid = eng_df[eng_df['published date'].notna()].copy()

# Apply the same filters as training
df_valid = df_valid[
    (df_valid['Price'] >= PRICE_FLOOR) &
    (df_valid['Price'] < PRICE_CAP) &
    (df_valid['Milleage'] >= 0) &
    (df_valid['Milleage'] < MILEAGE_CAP)
].copy()

actual_days = max(1, (df_valid['published date'].max() - df_valid['published date'].min()).days)
effective_split_days = min(TEST_SPLIT_DAYS, max(7, int(actual_days * 0.3)))
split_date = df_valid['published date'].max() - pd.Timedelta(days=effective_split_days)

train_df = df_valid[df_valid['published date'] <= split_date].copy()
test_df  = df_valid[df_valid['published date'] >  split_date].copy()

for col in CATEGORICAL_FEATURES:
    test_df[col] = test_df[col].astype('category')

X_test_holdout = test_df[ALL_FEATURES].copy()
y_test_holdout = test_df['Price'].values

# Fill NaNs using train set median (matches what train_model() does internally)
numeric_cols = [c for c in X_test_holdout.columns if c not in CATEGORICAL_FEATURES]
for col in numeric_cols:
    if X_test_holdout[col].isna().any():
        train_median = train_df[col].median()
        X_test_holdout[col].fillna(train_median if pd.notna(train_median) else 0, inplace=True)

print(f"[OK] Train set      : {len(train_df):,} samples")
print(f"[OK] X_test_holdout : {X_test_holdout.shape}")
print(f"[OK] y_test_holdout : {len(y_test_holdout):,} samples")
print(f"[OK] Price range    : LKR {y_test_holdout.min():,.0f} – {y_test_holdout.max():,.0f}")

[OK] X_test_holdout : (6919, 24)
[OK] y_test_holdout : 6,919 samples
[OK] Price range    : LKR 111,111 – 29,900,000


In [53]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd

# ── Safety check ──────────────────────────────────────────────────
assert lgbm_model is not None,    "[ERROR] lgbm_model is None — run train_model() first."
assert len(X_test_holdout) > 0,   "[ERROR] X_test_holdout is empty — run split recreation first."
assert len(y_test_holdout) > 0,   "[ERROR] y_test_holdout is empty — run split recreation first."

print("  CLASSIFICATION REPORT — VEHICLE PRICE TIERS")


# ── 1. Generate predictions ───────────────────────────────────────
y_actual = y_test_holdout
y_pred   = exp_target(lgbm_model.predict(X_test_holdout))

# Clamp predictions to valid price range (model can extrapolate beyond training)
y_pred = np.clip(y_pred, PRICE_FLOOR, PRICE_CAP)

print(f"\n  Test set : {len(y_actual):,} samples")
print(f"  Actual   : LKR {y_actual.min():,.0f} – {y_actual.max():,.0f}  (median {np.median(y_actual):,.0f})")
print(f"  Predicted: LKR {y_pred.min():,.0f} – {y_pred.max():,.0f}  (median {np.median(y_pred):,.0f})")

# ── 2. Define price tier boundaries ──────────────────────────────
q33 = np.percentile(y_actual, 33)
q67 = np.percentile(y_actual, 67)

print(f"\n  Price tier boundaries:")
print(f"  {'Budget':<12}: < LKR {q33:>14,.0f}")
print(f"  {'Mid-Range':<12}: LKR {q33:>14,.0f}  –  {q67:>14,.0f}")
print(f"  {'Premium':<12}: > LKR {q67:>14,.0f}")

# ── 3. Convert prices → tier labels ──────────────────────────────
labels     = ['Budget', 'Mid-Range', 'Premium']
y_act_tier = np.digitize(y_actual, [q33, q67])
y_prd_tier = np.digitize(y_pred,   [q33, q67])

actual_dist = pd.Series(y_act_tier).value_counts().sort_index()
print(f"\n  Actual tier distribution:")
for i, label in enumerate(labels):
    count = actual_dist.get(i, 0)
    pct   = count / len(y_actual) * 100
    print(f"    {label:<12}: {count:>6,} samples  ({pct:.1f}%)")

# ── 4. Classification report ──────────────────────────────────────

print("  Per-Tier Precision / Recall / F1")

print(classification_report(
    y_act_tier, y_prd_tier,
    target_names=labels,
    digits=3,
))

# ── 5. Confusion matrix ───────────────────────────────────────────
print("  Confusion Matrix  (rows = Actual | cols = Predicted)")
print()
cm = confusion_matrix(y_act_tier, y_prd_tier)

print(f"  {'':14}" + "".join(f"{l:>12}" for l in labels))

for i, row_label in enumerate(labels):
    print(f"  {row_label:<14}" + "".join(f"{cm[i][j]:>12,}" for j in range(len(labels))))

print()
print("  Per-tier accuracy:")
for i, label in enumerate(labels):
    row_total = cm[i].sum()
    correct   = cm[i][i]
    tier_acc  = correct / row_total * 100 if row_total > 0 else 0
    print(f"    {label:<12}: {tier_acc:>6.1f}%  ({correct:,} correct, {row_total - correct:,} misclassified)")

# ── 6. Tier-level MAPE ────────────────────────────────────────────

print("  Price Accuracy Within Each Tier  (MAPE)")

print(f"  {'Tier':<14}  {'MAPE':>8}  {'MAE (LKR)':>16}  {'Samples':>9}")

for i, label in enumerate(labels):
    mask = y_act_tier == i
    n    = mask.sum()
    if n == 0:
        print(f"  {label:<14}  {'N/A':>8}  {'N/A':>16}  {0:>9,}")
        continue
    tier_mape = np.mean(np.abs((y_actual[mask] - y_pred[mask]) / y_actual[mask])) * 100
    tier_mae  = np.mean(np.abs(y_actual[mask] - y_pred[mask]))
    print(f"  {label:<14}  {tier_mape:>7.2f}%  {tier_mae:>16,.0f}  {n:>9,}")

# ── 7. Overall summary ────────────────────────────────────────────

overall_acc   = (y_act_tier == y_prd_tier).mean() * 100
adj_1tier     = (np.abs(y_act_tier - y_prd_tier) <= 1).mean() * 100
correct_count = (y_act_tier == y_prd_tier).sum()
print(f"  Overall Tier Accuracy    : {overall_acc:>7.2f}%")
print(f"  Within-1-Tier Accuracy   : {adj_1tier:>7.2f}%")
print(f"  Correctly Classified     : {correct_count:>7,} / {len(y_actual):,} vehicles")



  CLASSIFICATION REPORT — VEHICLE PRICE TIERS

  Test set : 6,919 samples
  Actual   : LKR 111,111 – 29,900,000  (median 6,290,000)
  Predicted: LKR 693,897 – 30,000,000  (median 6,121,916)

  Price tier boundaries:
  Budget      : < LKR      4,468,800
  Mid-Range   : LKR      4,468,800  –       8,250,000
  Premium     : > LKR      8,250,000

  Actual tier distribution:
    Budget      :  2,283 samples  (33.0%)
    Mid-Range   :  2,339 samples  (33.8%)
    Premium     :  2,297 samples  (33.2%)
  Per-Tier Precision / Recall / F1
              precision    recall  f1-score   support

      Budget      0.907     0.972     0.938      2283
   Mid-Range      0.878     0.868     0.873      2339
     Premium      0.959     0.902     0.930      2297

    accuracy                          0.914      6919
   macro avg      0.915     0.914     0.914      6919
weighted avg      0.915     0.914     0.914      6919

  Confusion Matrix  (rows = Actual | cols = Predicted)

                      Budget 